In [1]:
# Note: 
# 1. You need to be connected to the HKU network or HKU VPN to access the chenlin04 server.
# 2. Do not upload chenlin04_student_login.txt to any public repository, such as GitHub and LLMs.
# 3. Contact Boao (boaozhan@connect.hku.hk) for any questions related to accessing/using the LinkUp data.

In [2]:
import pandas as pd
import requests
import clickhouse_connect
from collections import namedtuple

In [3]:
class SimpleResult:
    def __init__(self, json_data):
        self.result_rows = json_data.get('data', [])
        self.column_names = [meta['name'] for meta in json_data.get('meta', [])]

class SimpleClient:
    def __init__(self, host, username, password, **kwargs):
        self.base_url = f"http://{host}:8123/"
        self.auth = {
            'user': username,
            'password': password
        }
        
    def query(self, query_string):
        params = self.auth.copy()
        # Use JSON format to get metadata (column names) easily
        params['query'] = query_string + " FORMAT JSON"
        
        response = requests.get(self.base_url, params=params)
        
        if response.status_code != 200:
            raise Exception(f"Query failed: {response.text}")
            
        return SimpleResult(response.json())

# Initialize the client
with open('chenlin04_student_login.txt', 'r') as f:
    conn_info = eval(f.read())
    try:
        # Try to connect with clickhouse first
        client = clickhouse_connect.get_client(**conn_info)
        print("Connected using standard clickhouse_connect")
    except Exception as e:
        # If it doesn't work out, use requests
        print(f"Standard connection failed: {e}")
        print("Falling back to SimpleClient (lightweight mode)...")
        client = SimpleClient(**conn_info)
        print("Connected using SimpleClient")

Standard connection failed: Received ClickHouse exception, code: 75, server response: Code: 75. DB::ErrnoException: Cannot write to file /var/lib/clickhouse/tmp/tmpaf919bc1-cb96-4f78-8749-067d72c2b571: , errno: 28, strerror: No space left on device. (CANNOT_WRITE_TO_FILE_DESCRIPTOR)
Total space: 207.13 GiB
Available space: 0.00 B
Total inodes: 13.84 million
Available inodes: 12.35 million
Mount point: /
Filesystem: /dev/mapper/ubuntu--vg-ubuntu--lv (version 26.2.1.248 (official build)) (for url http://chenlin04.fbe.hku.hk:8123)
Falling back to SimpleClient (lightweight mode)...
Connected using SimpleClient


In [4]:
res = client.query("""show tables from fina4359_linkup_202603""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,name
0,company_id_compustat_identifiers
1,compustat_na_quarterly_all_firms_since_2000
2,country_state_code_mapping
3,crsp_daily_ret_all_firms_since_2000
4,crsp_monthly_ret_all_firms_since_2000
...,...
378,firm_70345
379,firm_70437
380,firm_7126
381,firm_8417


In [5]:
# company_id_compustat_identifiers is the mapping file between company_id and standard compustat identifiers
# company_id is the unique identifier for each company in the linkup database. It's used within this database only.
# compustat identifiers include gvkey, cusip, and tic. They are standard identifiers used in the financial industry and can be used to link to other databases.
# naics is the industry classification code for the company. See more from https://www.naics.com.

res = client.query("""select * from fina4359_linkup_202603.company_id_compustat_identifiers limit 5""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,company_id,cusip,gvkey,tic,conm,naics
0,20238,898402102,4685,TRMK,TRUSTMARK CORP,522110
1,43730,881569107,170969,TSRO,TESARO INC,325414
2,44223,539830109,6774,LMT,LOCKHEED MARTIN CORP,336414
3,12903,10948C107,33637,BV,BRIGHTVIEW HOLDINGS,541320
4,17378,38268T103,20644,GPRO,GOPRO INC,333316


In [6]:
res = client.query("""select * from fina4359_linkup_202603.company_id_compustat_identifiers
                   where tic = 'NKE'""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,company_id,cusip,gvkey,tic,conm,naics
0,10467,654106103,7906,NKE,NIKE INC -CL B,316210


In [7]:
# The job postings of a company with company_id x can be accessed through the table fina4359_linkup_202603.firm_x
# hash: unique identifier for each job posting.
# onet: occupation code for the job posting. See more from https://www.onetonline.org.
# created, last_updated, last_seen, and missing: date on which the job posting is created, last updated, last seen, and not seen anymore by the crawler.

res = client.query("""select * from fina4359_linkup_202603.firm_10467 limit 5""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,hash,onet,country,state,zip,created,last_updated,updates,last_seen,missing,description
0,0489e44cfda10132c1061d8aeff1692d,15119909,1,24,97228,2016-10-08,None,0,2017-02-07,2017-02-09,"Become a Part of the NIKE, Inc. Team\n\nNIKE, ..."
1,094b60f6a344771862769e2570220e28,41203100,1,20,55122,2016-09-02,None,0,2017-02-07,2017-02-09,"Become a Part of the NIKE, Inc. Team\n\nNIKE, ..."
2,0c5fffb23fcf90bdb359d6f2046f0806,41203100,1,2,77429,2017-01-06,None,0,2017-01-29,2017-02-01,"Become a Part of the NIKE, Inc. Team\n\nNIKE, ..."
3,0a28eddba451425b131ff6d0413035f2,41203100,1,2,77502,2016-09-02,None,0,2017-02-07,2017-02-09,"Become a Part of the NIKE, Inc. Team\n\nNIKE, ..."
4,0b4fabe2406bdd38698a8c60a3e0e64d,27202100,1,4,10601,2016-10-04,None,0,2017-02-03,2017-02-05,Work Hard. Play Hard.


In [15]:
# A job from Nike's latest job postings.

res = client.query("""select * from fina4359_linkup_202603.firm_10467 order by created desc limit 1""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,hash,onet,country,state,zip,created,last_updated,updates,last_seen,missing,description
0,15df67fabcfac7c9ec29e13710ba6973,13111100,1,24,97075,2022-09-29,None,0,2022-09-29,None,Who we are looking for\n \nWe are looking for...


In [8]:
# A job from Nike's latest job postings.

res = client.query("""with T1 as (
                    select max(created) as latest_created from fina4359_linkup_202603.firm_10467
                   )
                   select description from fina4359_linkup_202603.firm_10467 
                   where created = (select latest_created from T1) 
                   limit 1""")
test_job = res.result_rows
test_job

[{'description': "About the Role &amp; Team\\n  \\nAs a Principal Quality Engineer, you will be responsible for designing and implementing the Quality Engineering capabilities for Nike's Planning and Manufacturing Products. You will work on defining strategy, roadmap and implement quality focused governance, KPI's and metrics.\\n  \\nPartner with different product teams in creation and implementation of test assets \\u2014 test plans, cases, data, and reports. As an Expert Quality Engineer, you will be responsible for strategy, planning, execution tracking and reporting to validate technology solutions as per quality standards. You will be working along the product and engineering teams implementing quality frameworks and agile practices to enable speed to market of technology products. You will be responsible to ensure projects meet quality standards by providing technical guidance in planning, designing, and completing testing and developing procedures relating to product quality on 

In [9]:
print(test_job[0]['description'].replace('\\n', '\n'))

About the Role &amp; Team
  
As a Principal Quality Engineer, you will be responsible for designing and implementing the Quality Engineering capabilities for Nike's Planning and Manufacturing Products. You will work on defining strategy, roadmap and implement quality focused governance, KPI's and metrics.
  
Partner with different product teams in creation and implementation of test assets \u2014 test plans, cases, data, and reports. As an Expert Quality Engineer, you will be responsible for strategy, planning, execution tracking and reporting to validate technology solutions as per quality standards. You will be working along the product and engineering teams implementing quality frameworks and agile practices to enable speed to market of technology products. You will be responsible to ensure projects meet quality standards by providing technical guidance in planning, designing, and completing testing and developing procedures relating to product quality on complex projects. You will 

In [10]:
# Mapping between country names and state codes. 0 means unknown.

res = client.query("""select * from fina4359_linkup_202603.country_state_code_mapping limit 5""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,country,state,country_name,state_name
0,0,0,None,None
1,0,1,None,CA
2,0,2,None,TX
3,0,3,None,FL
4,0,4,None,VA


In [11]:
# Compustat North America quarterly data for all firms available in WRDS since 2000. 
# See variable definitions from Compustat_Understanding_The_Data.pdf. 

res = client.query("""select * from fina4359_linkup_202603.compustat_na_quarterly_all_firms_since_2000 limit 5""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,conm,cusip,...,seqq,txditcq,txtq,xintq,xsgaq,capxy,fincfy,ivncfy,oancfy,prccq
0,I,USD,STD,INDL,C,ADCT.1,2000-01-31,1013,ADC TELECOMMUNICATIONS INC,000886309,...,1396.005,0.000,15.300,NaN,205.600,46.785,16.289,-35.692,41.582,65.9375
1,I,USD,STD,INDL,C,SERV.1,2000-01-31,1082,SERVIDYNE INC,81765M106,...,21.270,4.009,-0.119,1.374,2.169,9.597,-4.911,-3.077,5.032,3.5000
2,I,USD,STD,INDL,C,AIM.1,2000-01-31,1173,AEROSONIC CORP,008015307,...,12.644,0.155,0.064,0.136,1.839,0.558,0.528,-0.558,-0.724,10.9375
3,I,USD,STD,INDL,C,IDAI.,2000-01-31,1183,IDNA INC,45169P106,...,83.879,0.000,-3.286,NaN,NaN,1.091,-2.582,43.731,-18.925,0.8400
4,I,CAD,STD,INDL,C,AGR.7,2000-01-31,1189,AGRA INDUSTRIES LTD,008489502,...,363.297,0.000,4.205,4.766,NaN,17.181,32.244,-23.324,-21.550,11.1000


In [12]:
# CRSP monthly return data for all firms available in WRDS since 2000.
# DIVAMT: cash dividend amount.
# PRC: price at the end of the month.
# VOL: trading volume during the month.
# RET: monthly return. Calculated as (PRC - lag(PRC) + DIVAMT) / lag(PRC).
# ALTPRC: alternative price when PRC is not available.
# RETX: return without dividends, calculated as (PRC - lag(PRC)) / lag(PRC).

res = client.query("""select * from fina4359_linkup_202603.crsp_monthly_ret_all_firms_since_2000 limit 5""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,PERMNO,date,TICKER,COMNAM,PERMCO,CUSIP,DIVAMT,PRC,VOL,RET,ALTPRC,RETX
0,10001,20000131,EWST,ENERGY WEST INC,7953,36720410,NaN,8.12500,403,-0.044118,8.12500,-0.044118
1,10001,20000229,EWST,ENERGY WEST INC,7953,36720410,NaN,8.25000,222,0.015385,8.25000,0.015385
2,10001,20000331,EWST,ENERGY WEST INC,7953,36720410,0.12,-8.00000,723,-0.015758,-8.00000,-0.030303
3,10001,20000428,EWST,ENERGY WEST INC,7953,36720410,NaN,-8.09375,263,0.011719,-8.09375,0.011719
4,10001,20000531,EWST,ENERGY WEST INC,7953,36720410,NaN,-7.90625,221,-0.023166,-7.90625,-0.023166


In [17]:
# For each company that can be found in company_id_compustat_identifiers, what is their average atq in 2020?

res = client.query("""with compustat_2020 as (
                    select gvkey, atq from fina4359_linkup_202603.compustat_na_quarterly_all_firms_since_2000
                    where datadate >= '2020-01-01' and datadate <= '2020-12-31'
                   )
                   select c.company_id, avg(t.atq) as avg_atq_2020 
                   from compustat_2020 t
                   join fina4359_linkup_202603.company_id_compustat_identifiers c
                   on t.gvkey = c.gvkey
                   group by c.company_id""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

,company_id,avg_atq_2020
0,21744,1906.94925
1,22452,10540.75225
2,11440,2028.58175
3,26972,484.05150
4,20503,504744.00000
...,...,...
354,20531,1953.76500
355,53143,2467.79300
356,65176,162.48600
357,44919,8695.50000


In [18]:
# On top of the previous table, add another column showing the stock return in 2020-12.

res = client.query("""with compustat_2020 as (
                    select gvkey, atq from fina4359_linkup_202603.compustat_na_quarterly_all_firms_since_2000
                    where datadate >= '2020-01-01' and datadate <= '2020-12-31'
                   ),
crsp_2020 as (
                    select cusip, ret from fina4359_linkup_202603.crsp_monthly_ret_all_firms_since_2000
                    where date >= '2020-12-01' and date <= '2020-12-31'
                   )
                   select c.company_id, avg(t.atq) as avg_atq_2020, r.ret as dec_2020_return
                   from compustat_2020 t
                   join fina4359_linkup_202603.company_id_compustat_identifiers c
                   on t.gvkey = c.gvkey
                   left join crsp_2020 r
                   on c.cusip = r.cusip
                   group by c.company_id, r.ret""")
df_tables = pd.DataFrame(res.result_rows, columns=res.column_names)
df_tables

Exception: Query failed: Code: 47. DB::Exception: Unknown expression identifier `ret` in scope  crsp_2020 AS r. Maybe you meant: ['RET']. (UNKNOWN_IDENTIFIER) (version 26.2.1.248 (official build))
